In [96]:
import os
import json
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
with open("config/config.json","r") as f:
    config=json.load(f)
print(config["llm"])

{'openai_model': 'gpt-4o-mini', 'gemini_model': 'gemini-2.5-flash', 'temperature': 0.3}


In [97]:
df=pd.read_csv("data/transcripts.csv")
print("data loaded")
df.head()
#df.head(2)-wil print only 2 top records
#df.tail()-wl print last 5 records from csv

data loaded


,call_id,agent_name,transcript,expected_call_type
0,1,Alice,Customer: I was charged twice for my premium t...,billing
1,2,Bob,Customer: My claim has been pending for 2 week...,claims
2,3,Charlie,Customer: This is the third time I'm calling! ...,complaint
3,4,Diana,Customer: Can you explain my coverage details?...,general_query


In [98]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI


def load_llm(config):
    provider=os.getenv("LLM_PROVIDER")

    if provider=="openai":
        llm=ChatOpenAI(
            model=config["llm"]["openai_model"],
            temperature=config["llm"]["temperature"]
        )

    elif provider=="gemini":
        llm=ChatGoogleGenerativeAI(
            model=config["llm"]["gemini_model"],
            temperature=config["llm"]["temperature"]
        )

    else:
        raise ValueError("invalid LLM provider")
    return llm

llm=load_llm(config)
print(f"LLM initialized provider: {os.getenv('LLM_PROVIDER')}")

LLM initialized provider: openai


In [99]:
from urllib import response


response=llm.invoke("Say 'setup successful' in one short sentance")
print(f"LLM response : {response.content}")



LLM response : Setup successful!


In [100]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel,Field

#define output schema in a class

class ClasificationOutput(BaseModel):
    call_type: str=Field(description="Type of customer call")
    confidence:float=Field(description="confidence score between 0 and 1")

parser=PydanticOutputParser(pydantic_object=ClasificationOutput)


#prompt templet
prompt=PromptTemplate(
    template="""
    You are call classification assistant.

    Classify the following customer support transcript into one of follwoing categories:
    {labels}

    Transcript:
    {transcript}

    {format_instructions}
    """,
    input_variables=["transcript"],
    partial_variables={
        "format_instructions":parser.get_format_instructions(),
        "labels":config["classification"]["labels"]
    }
)

#create chain
classification_chain=prompt | llm | parser

#test chain with one transcript
sample_text=df.iloc[0]["transcript"]
result=classification_chain.invoke({
    "transcript": sample_text
})

print(f"classification result: \n{result}")

classification result: 
call_type='billing' confidence=0.95


In [101]:
from tqdm import tqdm

results=[]

for i, row in tqdm(df.iterrows(),total=len(df),desc="Classifying calls"):

    try:
        output=classification_chain.invoke({
            "transcript":row["transcript"]

        })
        results.append({
            "call_id":row["call_id"],
            "predicted_call_type":output.call_type,
            "confidence":output.confidence
        })
    except Exception as e:
        print(f"Error at row {i}{e}")
        results.append({
            "call_id":row["call_id"],
            "predicted_call_type":None,
            "confidence":None
        })

results_df=pd.DataFrame(results)
df=df.merge(results_df,on="call_id")
df.head()


Classifying calls: 100%|██████████| 4/4 [00:03<00:00,  1.12it/s]


,call_id,agent_name,transcript,expected_call_type,predicted_call_type,confidence
0,1,Alice,Customer: I was charged twice for my premium t...,billing,billing,0.95
1,2,Bob,Customer: My claim has been pending for 2 week...,claims,claims,0.95
2,3,Charlie,Customer: This is the third time I'm calling! ...,complaint,complaint,0.90
3,4,Diana,Customer: Can you explain my coverage details?...,general_query,general_query,0.95


In [102]:
#-----Step1:define routing logic-----------

def route_call(call_type):
    if call_type=="billing":
        return ["knowledge_accuracy","resulution_quality"]

    elif call_type=="claims":
        return ["knowledge_accuracy","resulution_quality"]
    
    elif call_type=="complaint":
        return ["tone_empathy","resulution_quality"]

    elif call_type=="general_query":
            return ["knowledge_accuracy"]

#fallback
    else:
          return ["knowledge_accuracy"]

df["evaluation_plan"]=df["predicted_call_type"].apply(route_call)

print("------routing applied------")
print(df[["call_id","predicted_call_type","evaluation_plan"]])
    

------routing applied------
   call_id predicted_call_type                           evaluation_plan
0        1             billing  [knowledge_accuracy, resulution_quality]
1        2              claims  [knowledge_accuracy, resulution_quality]
2        3           complaint        [tone_empathy, resulution_quality]
3        4       general_query                      [knowledge_accuracy]


In [103]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel,Field

#define an output for tone evaluation

class ToneEvaluation(BaseModel):
    score: int=Field(description="Score between 1 and 5")
    reasoning: str=Field(description="Explaination of score")

tone_parser=PydanticOutputParser(pydantic_object=ToneEvaluation)

tone_prompt=PromptTemplate(template=""" 
You are a QA evaluator for customer support calls.

Evaluate the agent's tone and empathy in the following transcript.

Consider:
-Did the agent acknowledge the customer's issue?
-Was the tone polite and proffessional enough?
-Did agent show empathy?

Transcript:
{transcript}

{format_instructions}
""",
input_variables=["transcript"],
partial_variables={
    "format_instructions":tone_parser.get_format_instructions()
}

)

tone_chain=tone_prompt | llm | tone_parser

sample_text=df[df["predicted_call_type"]=="complaint"].iloc[0]["transcript"]
result=tone_chain.invoke({
    "transcript": sample_text
})

print(f"Tone evaluation result: {result}")


Tone evaluation result: score=3 reasoning="The agent acknowledged the customer's frustration by apologizing for the inconvenience, which shows some level of empathy. However, the response was quite brief and did not fully address the customer's repeated issues or express a deeper understanding of their situation. The tone was polite and professional, but it lacked warmth and a more personalized touch that could have enhanced the empathetic response."


In [104]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel,Field

#define an output for resolution evaluation

class ResulutionEvaluation(BaseModel):
    score: int=Field(description="Score between 1 and 5")
    reasoning: str=Field(description="Explaination of score")

resolution_parser=PydanticOutputParser(pydantic_object=ResulutionEvaluation)

resolution_prompt=PromptTemplate(template=""" 
You are a QA evaluator for customer support calls.

Evaluate the resulution quality of the agent

Consider:
-Did the agent fully resolved customer's issue?
-Was next step clearly communicated
-Did agent confirm resolution before ending?

Transcript:
{transcript}

{format_instructions}
""",
input_variables=["transcript"],
partial_variables={
    "format_instructions":resolution_parser.get_format_instructions()
}

)

resolution_chain=resolution_prompt | llm | resolution_parser

sample_text=df.iloc[0]["transcript"]
result=resolution_chain.invoke({
    "transcript": sample_text
})
print(f"Resolution evaluation result: {result}")


Resolution evaluation result: score=1 reasoning="The agent did not fully resolve the customer's issue as the transcript ends abruptly without confirming the resolution or providing next steps. There is no indication that the agent addressed the double charge issue."


In [105]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel,Field

#define an output for knowledge evaluation

class knowledgeEvaluation(BaseModel):
    score: int=Field(description="Score between 1 and 5")
    reasoning: str=Field(description="Explaination of score")

knowledge_parser=PydanticOutputParser(pydantic_object=knowledgeEvaluation)

knowledge_prompt=PromptTemplate(template=""" 
You are a QA evaluator for customer support calls.

Evaluate the agent's knowledge accuracy and clarity

Consider:
-Did the agent provide correct and relivant information?
-Was the explaination clear and easy to understand?
-Did agent avoid vague or misleading statements?

IMPORTANT:
-If the transcript does not contain enough information, give a moderate score(2 or 3) and explain why.

Transcript:
{transcript}

{format_instructions}
""",
input_variables=["transcript"],
partial_variables={
    "format_instructions":knowledge_parser.get_format_instructions()
}

)

knowledge_chain=knowledge_prompt | llm | knowledge_parser

sample_text=df.iloc[0]["transcript"]
result=knowledge_chain.invoke({
    "transcript": sample_text
})
print(f"knowledge evaluation result: {result}")


knowledge evaluation result: score=2 reasoning="The transcript does not provide enough information to evaluate the agent's knowledge accuracy and clarity effectively. The agent's response is limited to checking the issue without any specific information or explanation given to the customer. Therefore, while the agent is taking steps to address the concern, there is insufficient detail to assess the correctness or clarity of the information provided."


In [106]:
from tqdm import tqdm

def run_evaluation(transcript,eval_plan):
    results={}
    if "tone_empathy" in eval_plan:
        try:
            tone_result=tone_chain.invoke({"transcript":transcript})
            results["tone"]=tone_result.model_dump()
        except Exception as e:
            results["tone"]={"error": str(e)}

    if "knowledge_accuracy" in eval_plan:
            try:
                knowledge_result=knowledge_chain.invoke({"transcript":transcript})
                results["knowledge"]=knowledge_result.model_dump()
            except Exception as e:
                results["knowledge"]={"error": str(e)}


    if "resolution_quality" in eval_plan:
            try:
                resolution_result=resolution_chain.invoke({"transcript":transcript})
                results["resolution"]=resolution_result.model_dump()
            except Exception as e:
                results["resolution"]={"error": str(e)}
    return results

evaluation_outputs=[]

for i,row in tqdm(df.iterrows(), total=len(df),desc="Running Evaluations"):
     output=run_evaluation(row["transcript"],row["evaluation_plan"])

     evaluation_outputs.append({
          "call_id":row["call_id"],
          "evaluation_output":output
     })

#convert to dataframe
eval_df=pd.DataFrame(evaluation_outputs)

#merge
df=df.merge(eval_df,on="call_id")

print("\n Evaluation completed")


#show example
import pprint
pprint.pprint(df.iloc[0]["evaluation_output"])
    


Running Evaluations: 100%|██████████| 4/4 [00:05<00:00,  1.49s/it]


 Evaluation completed
{'knowledge': {'reasoning': 'The transcript does not provide enough '
                            "information about the agent's response after "
                            "checking the customer's issue. While the agent "
                            'initiated the process to check the double charge, '
                            'there is no indication of whether they provided '
                            'correct and relevant information or if their '
                            'explanation was clear. Therefore, a moderate '
                            'score is appropriate.',
               'score': 2}}


In [107]:
# --- Step 1: Schema ---
class FinalReport(BaseModel):
    summary: str = Field(description="Overall evaluation summary")
    recommendations: list[str] = Field(description="List of actionable improvements")

final_parser = PydanticOutputParser(pydantic_object=FinalReport)

# --- Step 2: Prompt ---
final_prompt = PromptTemplate(
    template="""
You are a QA manager reviewing customer support calls.

Based on the evaluation results below, generate:

1. A concise summary of the agent's performance
2. A list of actionable recommendations for improvement

Evaluation Data:
{evaluation_output}

IMPORTANT:
- Be specific and practical
- Do not repeat scores
- Focus on improvement

{format_instructions}
""",
    input_variables=["evaluation_output"],
    partial_variables={
        "format_instructions": final_parser.get_format_instructions()
    }
)


# --- Step 3: Chain ---
final_chain = final_prompt | llm | final_parser

# --- Step 4: Test on one row
sample_eval = df.iloc[0]["evaluation_output"]

result = final_chain.invoke({
    "evaluation_output": sample_eval
})

print("✅ Final QA Report:")
print(result)



✅ Final QA Report:
summary="The agent demonstrated an initial effort to address the customer's issue regarding a double charge but lacked clarity and completeness in their response. The evaluation indicates that the agent's knowledge and communication could be improved to enhance customer understanding and satisfaction." recommendations=['Provide clear and detailed explanations when checking customer issues.', 'Enhance product knowledge to ensure accurate information is conveyed to customers.', 'Practice active listening to better understand customer concerns before responding.', 'Engage in role-playing scenarios to improve clarity and confidence in communication.', 'Seek feedback from peers or supervisors on call handling to identify areas for improvement.']


In [108]:
from tqdm import tqdm
 
final_outputs = []
 
for i, row in tqdm(df.iterrows(), total=len(df), desc="Generating Final Reports"):
    try:
        result = final_chain.invoke({
            "evaluation_output": row["evaluation_output"]
        })
 
        final_outputs.append({
            "call_id": row["call_id"],
            "summary": result.summary,
            "recommendations": result.recommendations
        })
 
    except Exception as e:
        print(f"❌ Error at row {i}: {e}")
        final_outputs.append({
            "call_id": row["call_id"],
            "summary": None,
            "recommendations": None
        })
 
# Convert to DataFrame
final_df = pd.DataFrame(final_outputs)
 
# Merge
df = df.merge(final_df, on="call_id")
 
print("\n✅ Final Reports Generated\n")
 
# Show final result
df[[
    "call_id",
    "predicted_call_type",
    "evaluation_output",
    "summary",
    "recommendations"
]]
 

Generating Final Reports: 100%|██████████| 4/4 [00:07<00:00,  1.88s/it]


✅ Final Reports Generated



,call_id,predicted_call_type,evaluation_output,summary,recommendations
0,1,billing,"{'knowledge': {'score': 2, 'reasoning': 'The t...",The agent demonstrated an initial effort to ad...,[Provide more detailed explanations when addre...
1,2,claims,"{'knowledge': {'score': 2, 'reasoning': 'The t...",The agent demonstrated limited knowledge durin...,[Enhance training on claim processes to provid...
2,3,complaint,"{'tone': {'score': 3, 'reasoning': 'The agent ...",The agent demonstrated a polite tone and ackno...,[Enhance empathetic communication by acknowled...
3,4,general_query,"{'knowledge': {'score': 3, 'reasoning': 'The t...",The agent demonstrated a willingness to explai...,[Enhance training on product knowledge to ensu...


In [109]:
accuracy=(df["expected_call_type"]==df["predicted_call_type"]).mean()
print(f"classification accuracy: {accuracy}")

classification accuracy: 1.0


In [110]:
df.to_excel("data/output.xlsx",index=False)